# Model evaluation workflow designed for WildObs Image Management Platform

## Description
- This script can be used for benchmarking an ai species recognition model with a local dataset to get an independent assessment of Recall, Precision and F1 Score for a given location
- The purpose is to evaluate the most suitable or best performing model for a given location and inform an appropriate validation workflow to ensure accuracy requirements are met

## Setup instructions:
1) Prepare the testing dataset. Start by organising camera trap images into folders by species on your local computer. The quality of the testing dataset will determine the accuracy of the report generated. Here are a few tips:
- Use a representive number of images of each species e.g. at least 1000 if possible
- Avoid using images that have been used as part of the model training dataset as these will create a biased result
- If possible, select a random subset of testing images from a larger pool aiming to get a wide range of images over space and time
- For the purpose of calculating Recall, include species that are relevant to your monitoring program
- For the the purpose of calculating Precision, also include images of other species that are commonly detected on the cameras at that location even if they are not relevant to your monitoring. Also include some blank images. Below is an example breakdown:

| Species         | # images | Reason for inclusion                     |
|:----------------|:---------|:-----------------------------------------|
| Feral Cat       | 1000     | Target species in monitoring program     |
| Red Fox         | 1000     | Target species in monitoring program     |
| European Rabbit | 1000     | Target species in monitoring program     |
| Kangaroo        | 1000     | Non-target species but abundant at this location. Impact on Precision. |
| Emu             | 1000     | Non-target species but abundant at this location. Impact on Precision. |
| Blank           | 1000     | Impact on Precision. |

2) Establish new Project/s in the WildObs WIMP for benchmarking purposes:
-  You will need a separate Project for each model you are testing. 
-  Name the project based on the model that will be tested e.g. "Model benchmark testing: WildObs National".
-  Set the Sequence cutoff to 0 seconds. This aims to prevent the software from creating sequences so that each image is assessed independently.
-  Define Tags in the project based on the scientific names of the species you are testing. Tags need to match with the species names used in the WIMP.
-  Configure the project to use the model you want to test
3) Create Deployments:
- You will need to create a Deployment for each of the species you are testing.
- Upload the relevant images into each deployment.
- Use the tags created earlier to assign to the deployment so you know which species it is supposed to be. This will be used by the script to match the species to the model predictions
- Repeat for each Project, uploading the same set of images to each
4) Run the uploaded images through the AI species recognition model
5) Once model processing is complete for all deployments, export the project data in Camtrap DP format
6) Download and extract (unzip) the exported data to a folder on your local computer
7) Use the folder path as input to this script

## 1. Setup

In [26]:
import sys
from pathlib import Path

# make the project's src package importable when running from the scripts folder
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "scripts" else Path.cwd()
sys.path.insert(0, str(PROJECT_ROOT))

# drop cached project modules so re-running the notebook always uses the current code
for name in [m for m in sys.modules if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

from src.config import BenchmarkConfig
from src.notebook_display import display_report
from src.pipeline import run_benchmark
from src.templating import list_sections, list_templates, resolve_template

## 2. Choose a report template
Templates live in the `templates` folder as Markdown files. Run the cell below to see the available templates and the sections within each that can be switched off. To make your own, copy an existing template, edit it, and refer to it by name in the config below. See `templates/README.md`.

In [27]:
for name, description in list_templates(PROJECT_ROOT / "templates").items():
    print(f"{name}: {description}")
    print(f"    sections: {', '.join(list_sections(resolve_template(name, PROJECT_ROOT / 'templates')))}")

full: Complete report including purpose, limitations, metric explanations and all results.
    sections: Test_Details, Purpose, Limitations, Metric_Definitions, Terms_And_Interpretation, Confusion_Matrix_Explanation, Results, Optimal_Thresholds, Prediction_Distribution, Full_Confusion_Matrices
minimal: Focused report with test details and results only; no interpretive or explanatory text.
    sections: Test_Details, Results, Optimal_Thresholds, Prediction_Distribution, Full_Confusion_Matrices


## 3. User configuration

In [29]:
config = BenchmarkConfig(
    # Folder name of the extracted Camtrap DP export (inside Input_Data)
    input_camtrap_folder_name="model-testing--edgbaston-reserve--model-awc135-20261007043009",

    # Details shown in the report
    data_source_location="Edgbaston Reserve, QLD",
    data_collation_process="""
""",

    thresholds=[0.5, 0.6, 0.7, 0.8, 0.9],
    export_errors=True,  # also save a CSV of misclassified images

    # Report format: "full", "minimal", another template name, or a path to your own .md template
    template="full",
    # Optional: leave out named sections of the chosen template, e.g. ["Limitations", "Full_Confusion_Matrices"]
    exclude_sections=[],

    log_level="INFO",  # use "DEBUG" for more detail on screen (the log file always records DEBUG)
)

## 4. Run
Outputs are saved to `Output_Reports` and a detailed log to `logs`.

In [30]:
run = run_benchmark(config)
print(f"Report saved to: {run.report_path}")

2026-10-07 21:05:21,015 | INFO     | src.logging_setup | Logging to C:\GitHub_Repos\model-benchmark-testing-workflow\logs\benchmark_20261007_210521.log
2026-10-07 21:05:21,016 | INFO     | src.pipeline | Starting benchmark run 20261007_210521
2026-10-07 21:05:21,019 | INFO     | src.data_loader | Loading Camtrap DP data from C:\GitHub_Repos\model-benchmark-testing-workflow\Input_Data\model-testing--edgbaston-reserve--model-awc135-20261007043009
2026-10-07 21:05:21,150 | INFO     | src.data_loader | Loaded 17014 observations, 8190 media files, 37 deployments
2026-10-07 21:05:21,256 | INFO     | src.data_loader | Model: AWC 135 | 26768 images | 37 species/labels in deployment tags
2026-10-07 21:05:21,261 | INFO     | src.metrics | Computing metrics for 37 species x 5 thresholds
2026-10-07 21:05:22,065 | INFO     | src.report | Exported 17301 misclassified images to C:\GitHub_Repos\model-benchmark-testing-workflow\Output_Reports\Misclassified_Images_AWC 135_20261007_210521.csv
2026-10-07 

Report saved to: C:\GitHub_Repos\model-benchmark-testing-workflow\Output_Reports\Model_Benchmark_Test_Report_AWC 135_full_20261007_210521.html


## 5. View the report (optional)

In [31]:
display_report(run.report_path)